# Tutorial 4 — hERG Cardiotoxicity Prediction
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

hERG (human Ether-à-go-go-Related Gene) encodes a cardiac potassium channel. Blocking it causes QT prolongation and potentially fatal arrhythmia — it has caused multiple drug withdrawals. Predicting hERG blockade early saves enormous resources.

This tutorial builds on my *Chemistry* 2022 paper combining physicochemical features with ML.

In [ ]:
!pip install rdkit scikit-learn pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt

# Known hERG blockers (1) and non-blockers (0)
# Source: ChEMBL / literature
herg_data = [
    ("Terfenadine",  "OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1", 1),
    ("Cisapride",    "CCOC(=O)c1cc2cc(OC)c(OC)cc2[nH]1", 1),
    ("Sertindole",   "Clc1ccc2c(c1)n(CCN1CCC(=C3c4cc(F)ccc4NC3=O)CC1)c(=O)n2", 1),
    ("Astemizole",   "Fc1ccc(cc1)CCN1CCC(CC1)Nc1nc2ccccc2n1Cc1ccc(OC)cc1", 1),
    ("Dofetilide",   "CN(CCOc1ccc(NS(=O)(=O)c2ccc(NC)cc2)cc1)S(=O)(=O)c1ccc(N)cc1", 1),
    ("Amiodarone",   "CCCc1oc2ccccc2c1C(=O)c1ccc(OCCN(CC)CC)c(I)c1I", 1),
    ("Aspirin",      "CC(=O)Oc1ccccc1C(=O)O", 0),
    ("Metformin",    "CN(C)C(=N)NC(=N)N", 0),
    ("Caffeine",     "Cn1cnc2c1c(=O)n(C)c(=O)n2C", 0),
    ("Penicillin G", "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O", 0),
    ("Atenolol",     "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1", 0),
    ("Metoprolol",   "COCCC(=O)c1ccc(OCC(O)CNC(C)C)cc1", 0),
]

def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 1024))
    pc = [
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol), rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol), rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        Descriptors.MolMR(mol), Descriptors.NumValenceElectrons(mol),
    ]
    return np.array(fp + pc)

X = np.array([featurize(s) for _, s, _ in herg_data])
y = np.array([label for _, _, label in herg_data])
names_list = [n for n, _, _ in herg_data]

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
for name, clf in [
    ("Random Forest", RandomForestClassifier(100, random_state=42)),
    ("Gradient Boost", GradientBoostingClassifier(100, random_state=42)),
]:
    scores = cross_val_score(clf, X_s, y, cv=cv, scoring="roc_auc")
    print(f"{name:20s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}")

## Feature importance analysis

In [ ]:
rf = RandomForestClassifier(200, random_state=42)
rf.fit(X_s, y)

# Feature names: first 1024 are fingerprint bits, then physicochemical
pc_names = ["MW","LogP","TPSA","HBD","HBA","RotBonds","ArRings","MR","ValElec"]
feat_names = [f"FP_{i}" for i in range(1024)] + pc_names
importances = rf.feature_importances_

# Show top 15 overall features
top_idx = np.argsort(importances)[::-1][:15]
top_names = [feat_names[i] for i in top_idx]
top_vals  = importances[top_idx]

# Physicochemical only
pc_imp = {pc_names[i]: importances[1024+i] for i in range(len(pc_names))}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.barh(top_names[::-1], top_vals[::-1], color="#1565c0")
ax1.set_xlabel("Importance")
ax1.set_title("Top 15 features (RF)")

ax2.barh(list(pc_imp.keys()), list(pc_imp.values()), color="#00897b")
ax2.set_xlabel("Importance")
ax2.set_title("Physicochemical features only")
plt.tight_layout()
plt.savefig("herg_importance.png", dpi=150)
plt.show()
print("\nPhysicochemical importances:", {k: f"{v:.4f}" for k,v in pc_imp.items()})

## Key takeaways
- LogP and molecular weight are the most predictive physicochemical features for hERG
- Combining fingerprints with physicochemical descriptors outperforms either alone
- For production models, use larger datasets from ChEMBL (search hERG, IC50 assays)
- See: Goel H, Yu W, MacKerell Jr. AD. *Chemistry* 2022, 4, 630–646